# Italian Restaurant in Pure Python
## Five Exact Methods for Book Problem 2.9

This notebook presents five exact approaches for the Italian restaurant problem in pure Python:

1. Naive enumeration
2. Backtracking with pruning
3. Constraint-driven reduced search
4. Dynamic programming
5. Branch and Bound

We solve:
- the base production model from book section `2.9`
- the student variation that forces at least 3 plates of each recipe


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from itertools import product
from time import perf_counter


In [2]:
def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def total_profit(problem, values):
    return problem["base_profit"] + sum(value * profit for value, profit in zip(values, problem["profits"]))


def choose_better(problem, left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_profit = total_profit(problem, left)
    right_profit = total_profit(problem, right)
    if right_profit > left_profit:
        return right
    if right_profit < left_profit:
        return left
    return right if tuple(right) < tuple(left) else left


def is_feasible(problem, values):
    for resource_index, capacity in enumerate(problem["capacities"]):
        used = sum(problem["usage"][dish][resource_index] * values[dish] for dish in range(len(values)))
        if used > capacity:
            return False
    return True


def restore_solution(problem, values):
    actual_values = [lower + value for lower, value in zip(problem["lower_bounds"], values)]
    result = {
        name: actual_values[index]
        for index, name in enumerate(problem["names"])
    }
    result["profit"] = total_profit(problem, values)
    return result


In [3]:
def normalize_problem(names, profits, usage, capacities, lower_bounds, upper_bounds):
    adjusted_capacities = capacities[:]
    base_profit = 0

    for dish, lower in enumerate(lower_bounds):
        base_profit += lower * profits[dish]
        for resource in range(len(capacities)):
            adjusted_capacities[resource] -= lower * usage[dish][resource]

    return {
        "names": names,
        "profits": profits,
        "usage": usage,
        "capacities": adjusted_capacities,
        "lower_bounds": lower_bounds,
        "upper_bounds": [upper - lower for upper, lower in zip(upper_bounds, lower_bounds)],
        "base_profit": base_profit,
    }


NAMES = ["tallarines", "ravioles", "lasagna", "corbatitas", "fideos"]
PROFITS = [50, 65, 40, 70, 55]
UPPER_BOUNDS = [15, 12, 32, 18, 10]
USAGE = [
    [1, 1, 0, 1],
    [4, 2, 0, 1],
    [2, 1, 1, 2],
    [0, 0, 2, 0],
    [1, 1, 0, 1],
]
CAPACITIES = [80, 75, 10, 30]

BASE_PROBLEM = normalize_problem(
    names=NAMES,
    profits=PROFITS,
    usage=USAGE,
    capacities=CAPACITIES,
    lower_bounds=[0, 0, 0, 0, 0],
    upper_bounds=UPPER_BOUNDS,
)

VARIANT_PROBLEM = normalize_problem(
    names=NAMES,
    profits=PROFITS,
    usage=USAGE,
    capacities=CAPACITIES,
    lower_bounds=[3, 3, 3, 3, 3],
    upper_bounds=UPPER_BOUNDS,
)

BASE_EXPECTED = {
    "tallarines": 8,
    "ravioles": 12,
    "lasagna": 0,
    "corbatitas": 5,
    "fideos": 10,
    "profit": 2080,
}

VARIANT_EXPECTED = {
    "tallarines": 3,
    "ravioles": 12,
    "lasagna": 3,
    "corbatitas": 3,
    "fideos": 9,
    "profit": 1755,
}


## Problem 1: Base Restaurant Model

The restaurant chooses daily production quantities for five recipes subject to:
- ingredient freshness limits
- maximum demand for each dish

All quantities are integers.


In [4]:
def solve_restaurant_naive(problem):
    best = None
    ranges = [range(limit + 1) for limit in problem["upper_bounds"]]
    for values in product(*ranges):
        if not is_feasible(problem, values):
            continue
        best = choose_better(problem, best, values)
    return restore_solution(problem, best)


def solve_restaurant_backtracking(problem):
    best = None

    def backtrack(index, current_values, used_resources):
        nonlocal best
        if index == len(problem["names"]):
            best = choose_better(problem, best, tuple(current_values))
            return

        optimistic = problem["base_profit"] + sum(
            current_values[i] * problem["profits"][i] for i in range(index)
        ) + sum(
            problem["upper_bounds"][i] * problem["profits"][i] for i in range(index, len(problem["names"]))
        )
        if best is not None and optimistic < total_profit(problem, best):
            return

        max_value = problem["upper_bounds"][index]
        for value in range(max_value, -1, -1):
            feasible = True
            next_resources = used_resources[:]
            for resource in range(len(problem["capacities"])):
                next_resources[resource] += problem["usage"][index][resource] * value
                if next_resources[resource] > problem["capacities"][resource]:
                    feasible = False
                    break
            if not feasible:
                continue
            current_values[index] = value
            backtrack(index + 1, current_values, next_resources)
        current_values[index] = 0

    backtrack(0, [0] * len(problem["names"]), [0] * len(problem["capacities"]))
    return restore_solution(problem, best)


def solve_restaurant_reduced_search(problem):
    best = None
    idx_t, idx_r, idx_l, idx_c, idx_f = 0, 1, 2, 3, 4

    for ravioles in range(problem["upper_bounds"][idx_r] + 1):
        for lasagna in range(problem["upper_bounds"][idx_l] + 1):
            for corbatitas in range(problem["upper_bounds"][idx_c] + 1):
                values = [0] * 5
                values[idx_r] = ravioles
                values[idx_l] = lasagna
                values[idx_c] = corbatitas

                if not is_feasible(problem, values):
                    continue

                remaining = []
                for resource in range(len(problem["capacities"])):
                    used = (
                        problem["usage"][idx_r][resource] * ravioles
                        + problem["usage"][idx_l][resource] * lasagna
                        + problem["usage"][idx_c][resource] * corbatitas
                    )
                    remaining.append(problem["capacities"][resource] - used)

                fideos = min(
                    problem["upper_bounds"][idx_f],
                    remaining[0],
                    remaining[1],
                    remaining[3],
                )
                values[idx_f] = fideos

                remaining_after_fideos = remaining[:]
                for resource in range(len(problem["capacities"])):
                    remaining_after_fideos[resource] -= problem["usage"][idx_f][resource] * fideos

                tallarines = min(
                    problem["upper_bounds"][idx_t],
                    remaining_after_fideos[0],
                    remaining_after_fideos[1],
                    remaining_after_fideos[3],
                )
                values[idx_t] = tallarines

                if not is_feasible(problem, values):
                    continue

                best = choose_better(problem, best, tuple(values))

    return restore_solution(problem, best)


def solve_restaurant_dp(problem):
    usage = problem["usage"]

    @lru_cache(maxsize=None)
    def dp(index, t_cap, o_cap, e_cap, p_cap):
        if index == len(problem["names"]):
            return 0, ()

        best_profit = -1
        best_values = None
        for value in range(problem["upper_bounds"][index] + 1):
            if usage[index][0] * value > t_cap:
                break
            if usage[index][1] * value > o_cap:
                continue
            if usage[index][2] * value > e_cap:
                continue
            if usage[index][3] * value > p_cap:
                continue
            suffix_profit, suffix_values = dp(
                index + 1,
                t_cap - usage[index][0] * value,
                o_cap - usage[index][1] * value,
                e_cap - usage[index][2] * value,
                p_cap - usage[index][3] * value,
            )
            total = value * problem["profits"][index] + suffix_profit
            candidate = (value,) + suffix_values
            if total > best_profit:
                best_profit = total
                best_values = candidate
            elif total == best_profit and candidate < best_values:
                best_values = candidate
        return best_profit, best_values

    _, values = dp(0, *problem["capacities"])
    return restore_solution(problem, values)


def solve_restaurant_branch_and_bound(problem):
    best = None
    stack = [(0, (0, 0, 0, 0, 0), tuple(problem["capacities"]), 0)]

    while stack:
        index, current_values, remaining_resources, current_profit = stack.pop()

        if index == len(problem["names"]):
            best = choose_better(problem, best, current_values)
            continue

        optimistic = current_profit
        for future_index in range(index, len(problem["names"])):
            local_cap = problem["upper_bounds"][future_index]
            for resource in range(len(problem["capacities"])):
                use = problem["usage"][future_index][resource]
                if use > 0:
                    local_cap = min(local_cap, remaining_resources[resource] // use)
            optimistic += local_cap * problem["profits"][future_index]
        if best is not None and problem["base_profit"] + optimistic < total_profit(problem, best):
            continue

        max_value = problem["upper_bounds"][index]
        for value in range(max_value, -1, -1):
            next_resources = list(remaining_resources)
            feasible = True
            for resource in range(len(problem["capacities"])):
                need = problem["usage"][index][resource] * value
                if need > next_resources[resource]:
                    feasible = False
                    break
                next_resources[resource] -= need
            if not feasible:
                continue
            next_values = list(current_values)
            next_values[index] = value
            stack.append(
                (
                    index + 1,
                    tuple(next_values),
                    tuple(next_resources),
                    current_profit + value * problem["profits"][index],
                )
            )

    return restore_solution(problem, best)


In [5]:
@timed
def solve_base_naive():
    return solve_restaurant_naive(BASE_PROBLEM)


@timed
def solve_base_backtracking():
    return solve_restaurant_backtracking(BASE_PROBLEM)


@timed
def solve_base_reduced_search():
    return solve_restaurant_reduced_search(BASE_PROBLEM)


@timed
def solve_base_dp():
    return solve_restaurant_dp(BASE_PROBLEM)


@timed
def solve_base_branch_and_bound():
    return solve_restaurant_branch_and_bound(BASE_PROBLEM)


In [6]:
base_naive_result, base_naive_time = solve_base_naive()
base_backtracking_result, base_backtracking_time = solve_base_backtracking()
base_reduced_result, base_reduced_time = solve_base_reduced_search()
base_dp_result, base_dp_time = solve_base_dp()
base_bb_result, base_bb_time = solve_base_branch_and_bound()

print("=== BASE RESTAURANT RESULTS ===")
print(f"Naive enumeration            -> {base_naive_result}, time = {base_naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {base_backtracking_result}, time = {base_backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {base_reduced_result}, time = {base_reduced_time:.8f} seconds")
print(f"Dynamic programming          -> {base_dp_result}, time = {base_dp_time:.8f} seconds")
print(f"Branch and Bound             -> {base_bb_result}, time = {base_bb_time:.8f} seconds")

assert base_naive_result == BASE_EXPECTED
assert base_backtracking_result == base_naive_result
assert base_reduced_result == base_naive_result
assert base_dp_result == base_naive_result
assert base_bb_result == base_naive_result
print("All five exact methods agree with the textbook optimum.")


=== BASE RESTAURANT RESULTS ===
Naive enumeration            -> {'tallarines': 8, 'ravioles': 12, 'lasagna': 0, 'corbatitas': 5, 'fideos': 10, 'profit': 2080}, time = 1.81497933 seconds
Backtracking with pruning    -> {'tallarines': 8, 'ravioles': 12, 'lasagna': 0, 'corbatitas': 5, 'fideos': 10, 'profit': 2080}, time = 0.02769971 seconds
Constraint-driven reduced search -> {'tallarines': 8, 'ravioles': 12, 'lasagna': 0, 'corbatitas': 5, 'fideos': 10, 'profit': 2080}, time = 0.01372854 seconds
Dynamic programming          -> {'tallarines': 8, 'ravioles': 12, 'lasagna': 0, 'corbatitas': 5, 'fideos': 10, 'profit': 2080}, time = 0.04112329 seconds
Branch and Bound             -> {'tallarines': 8, 'ravioles': 12, 'lasagna': 0, 'corbatitas': 5, 'fideos': 10, 'profit': 2080}, time = 0.00550438 seconds
All five exact methods agree with the textbook optimum.


## Problem 2: Student Variation with Minimum Daily Production

The student variation asks for at least `3` plates of every recipe.

The notebook handles that by subtracting the mandatory 3 plates from every recipe first, then solving the remaining exact integer problem with the same five methods.


In [7]:
@timed
def solve_variant_naive():
    return solve_restaurant_naive(VARIANT_PROBLEM)


@timed
def solve_variant_backtracking():
    return solve_restaurant_backtracking(VARIANT_PROBLEM)


@timed
def solve_variant_reduced_search():
    return solve_restaurant_reduced_search(VARIANT_PROBLEM)


@timed
def solve_variant_dp():
    return solve_restaurant_dp(VARIANT_PROBLEM)


@timed
def solve_variant_branch_and_bound():
    return solve_restaurant_branch_and_bound(VARIANT_PROBLEM)


In [8]:
variant_naive_result, variant_naive_time = solve_variant_naive()
variant_backtracking_result, variant_backtracking_time = solve_variant_backtracking()
variant_reduced_result, variant_reduced_time = solve_variant_reduced_search()
variant_dp_result, variant_dp_time = solve_variant_dp()
variant_bb_result, variant_bb_time = solve_variant_branch_and_bound()

print("=== MINIMUM-3-PLATES VARIATION RESULTS ===")
print(f"Naive enumeration            -> {variant_naive_result}, time = {variant_naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {variant_backtracking_result}, time = {variant_backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {variant_reduced_result}, time = {variant_reduced_time:.8f} seconds")
print(f"Dynamic programming          -> {variant_dp_result}, time = {variant_dp_time:.8f} seconds")
print(f"Branch and Bound             -> {variant_bb_result}, time = {variant_bb_time:.8f} seconds")

assert variant_naive_result == VARIANT_EXPECTED
assert variant_backtracking_result == variant_naive_result
assert variant_reduced_result == variant_naive_result
assert variant_dp_result == variant_naive_result
assert variant_bb_result == variant_naive_result
print("All five exact methods agree with the student variation optimum.")


=== MINIMUM-3-PLATES VARIATION RESULTS ===
Naive enumeration            -> {'tallarines': 3, 'ravioles': 12, 'lasagna': 3, 'corbatitas': 3, 'fideos': 9, 'profit': 1755}, time = 0.52624050 seconds
Backtracking with pruning    -> {'tallarines': 3, 'ravioles': 12, 'lasagna': 3, 'corbatitas': 3, 'fideos': 9, 'profit': 1755}, time = 0.00373800 seconds
Constraint-driven reduced search -> {'tallarines': 3, 'ravioles': 12, 'lasagna': 3, 'corbatitas': 3, 'fideos': 9, 'profit': 1755}, time = 0.00612533 seconds
Dynamic programming          -> {'tallarines': 3, 'ravioles': 12, 'lasagna': 3, 'corbatitas': 3, 'fideos': 9, 'profit': 1755}, time = 0.00140204 seconds
Branch and Bound             -> {'tallarines': 3, 'ravioles': 12, 'lasagna': 3, 'corbatitas': 3, 'fideos': 9, 'profit': 1755}, time = 0.00097296 seconds
All five exact methods agree with the student variation optimum.
